In [1]:
!pip install --upgrade pip

Defaulting to user installation because normal site-packages is not writeable


In [2]:
!pip -q install -U "transformers>=4.43" accelerate bitsandbytes sentencepiece tqdm

In [4]:
import os, json, re, random, glob
from typing import Optional, List, Dict, Any
from tqdm import tqdm
import torch

# ====== ĐƯỜNG DẪN DỮ LIỆU (SỬA CHO ĐÚNG) ======
# Nếu bạn để cùng thư mục notebook:
TRAIN_PATH = "ViMUNCH/vimunch_train.json"
DEV_PATH   = "ViMUNCH/vimunch_dev.json"
TEST_PATH  = "ViMUNCH/vimunch_test.json"

print("TRAIN_PATH:", TRAIN_PATH)
print("DEV_PATH  :", DEV_PATH)
print("TEST_PATH :", TEST_PATH)

# ====== MODEL ======
MODEL_NAME = "SeaLLMs/SeaLLMs-v3-7B-Chat"
TRUST_REMOTE_CODE = False

# ====== OUTPUT ======
RESULT_ROOT = "Result/SeaLLMs-v3-7B"
os.makedirs(RESULT_ROOT, exist_ok=True)

# ====== FIXED FEW-SHOT IDS (đảm bảo công bằng) ======
FEWSHOT_IDS = [4839, 758, 427, 8151, 7679]

# ====== LABEL SPACE ======
ALLOWED_TYPES = ["structural","ontological","orientational","emotional","cultural_folklore"]

# ====== INFERENCE CONFIG ======
torch.backends.cuda.matmul.allow_tf32 = True
DTYPE = torch.bfloat16
BATCH_SIZE = 32
MAX_NEW_TOKENS = 512
RETRY_MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.0
DO_SAMPLE = False

# chạy thử dev trước
DEV_LIMIT = 200     # None để chạy full 850
TEST_LIMIT = None   # None để chạy full 1701


TRAIN_PATH: ViMUNCH/vimunch_train.json
DEV_PATH  : ViMUNCH/vimunch_dev.json
TEST_PATH : ViMUNCH/vimunch_test.json


In [5]:
def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train = load_json(TRAIN_PATH)
dev   = load_json(DEV_PATH)
test  = load_json(TEST_PATH)

print("Sizes:", len(train), len(dev), len(test))
print("Keys:", list(train[0].keys()))

Sizes: 5950 850 1701
Keys: ['id', 'sentence', 'have_metaphor', 'interpretation', 'metaphor_types', 'metaphor_phrases', 'notes', 'scores', 'num_metaphor_types', 'num_metaphor_phrases', 'has_interpretation', 'has_scores', 'sentence_len', 'score_accuracy', 'score_clarity', 'score_naturalness', 'score_meaning', 'score_modality', 'score_implication', 'score_syntax', 'score_context', 'score_overall', 'score_quality']


In [6]:
def build_fewshot_from_ids(train: List[Dict[str, Any]], ids: List[int]):
    by_str = {str(r["id"]): r for r in train}
    few = []
    missing = []
    for _id in ids:
        r = by_str.get(str(_id))
        if r is None:
            missing.append(_id)
        else:
            few.append(r)
    if missing:
        raise ValueError(f"Missing few-shot IDs in train: {missing}")
    return few

fewshot_5 = build_fewshot_from_ids(train, FEWSHOT_IDS)
fewshot_ids = [r["id"] for r in fewshot_5]
print("Locked fewshot_ids:", fewshot_ids)

Locked fewshot_ids: [4839, 758, 427, 8151, 7679]


In [7]:
def gold_to_annotate_json(r: Dict[str, Any]) -> Dict[str, Any]:
    """Few-shot cho bước Task 1–3: KHÔNG đưa scores để model không học tự chấm."""
    have = int(r.get("have_metaphor", 0))

    phrases = []
    for sp in (r.get("metaphor_phrases") or []):
        phrases.append({
            "phrase": sp["phrase"],
            "start": int(sp["start"]),
            "end": int(sp["end"]),
        })

    types = [t for t in (r.get("metaphor_types") or []) if t in ALLOWED_TYPES]
    interp = (r.get("interpretation") or "").strip()

    if have == 0:
        return {
            "have_metaphor": 0,
            "metaphor_phrases": [],
            "metaphor_types": [],
            "interpretation": "",
            "scores": None
        }

    return {
        "have_metaphor": 1,
        "metaphor_phrases": phrases,
        "metaphor_types": types,
        "interpretation": interp,
        "scores": None
    }


def gold_to_judge_json(r: Dict[str, Any]) -> Dict[str, Any]:
    """Few-shot cho bước Task 4: chấm GOLD interpretation."""
    gold_interp = (r.get("interpretation") or "").strip()
    if not gold_interp:
        return {"scores": None}
    return {"scores": r.get("scores", None)}

In [8]:
def build_prompt(
    sentence: str,
    approach: str,
    fewshot_examples: Optional[List[Dict[str, Any]]] = None,
    *,
    mode: str = "annotate",              # "annotate" hoặc "judge"
    gold_interpretation: Optional[str] = None
) -> str:
    """
    annotate: Task 1–3 (model tự diễn giải), scores luôn null
    judge:    Task 4 (chấm GOLD interpretation), output chỉ có scores
    """

    # ===== Schema hints =====
    schema_hint_annotate = {
        "have_metaphor": 0,
        "metaphor_phrases": [{"phrase": "...", "start": 0, "end": 0}],
        "metaphor_types": ["emotional"],
        "interpretation": "...",
        "scores": None
    }

    schema_hint_judge = {
        "scores": {
            "accuracy": 0, "clarity": 0, "naturalness": 0,
            "meaning": 0, "implication": 0, "modality": 0, "syntax": 0, "context": 0,
            "overall": 0.0, "quality": 0.0
        }
    }

    types_str = json.dumps(ALLOWED_TYPES, ensure_ascii=False)

    # ====== ANNOTATE MODE (Task 1–3) ======
    if mode == "annotate":
        rules = f"""
Bạn là người gán nhãn ẩn dụ tiếng Việt. Trả về DUY NHẤT 1 JSON HỢP LỆ (không markdown, không giải thích, không lặp lại input).

ĐỊNH NGHĨA NGẮN:
Ẩn dụ là cách diễn đạt dựa trên ánh xạ khái niệm: dùng miền nguồn (cụ thể/quen thuộc) để nói miền mục tiêu (trừu tượng/khó nắm).

TASK 1) Nhận diện + trích xuất span
- have_metaphor: 1 nếu có ít nhất 1 ẩn dụ, ngược lại 0.
- metaphor_phrases: list các span chứa ẩn dụ.
  + start/end 0-based, end exclusive.
  + phrase PHẢI khớp đúng sentence[start:end].
  + Chỉ chọn phần mang nghĩa ẩn dụ (không chọn phần thuần nghĩa đen).
  + Ưu tiên span “đủ ý” tạo nên ẩn dụ.

TASK 2) Phân loại ẩn dụ (multi-label, cấp câu)
- metaphor_types: chọn 0 hoặc nhiều nhãn trong đúng tập sau: {types_str}
Gợi ý cực ngắn:
- structural: hiểu A theo cấu trúc của B
- orientational: dùng hướng không gian (lên/xuống, vào/ra...) để biểu đạt trạng thái/giá trị
- ontological: xem khái niệm trừu tượng như vật thể/sinh thể có thể “tương tác”
- emotional: dùng hình ảnh để gợi/biểu đạt cảm xúc
- cultural_folklore: gắn văn hóa dân gian/điển cố/ca dao-tục ngữ/biểu tượng Việt

TASK 3) Diễn giải (BẠN TỰ VIẾT)
- interpretation: viết lại cho tường minh hơn (KHÔNG chép lại câu gốc) nhưng KHÔNG đổi nghĩa, KHÔNG thêm ý và KHÔNG bớt ý.
- Viết tiếng Việt tự nhiên, ngắn gọn.
- KHÔNG dùng dấu nháy kép " trong interpretation (nếu cần trích dẫn dùng nháy đơn '...').

QUY TẮC NULL (bắt buộc):
- Nếu have_metaphor=0:
  metaphor_phrases=[], metaphor_types=[], interpretation="", scores=null
- Nếu have_metaphor=1:
  cố gắng điền metaphor_phrases và metaphor_types; interpretation không rỗng.
  scores LUÔN = null (không tự chấm điểm ở bước này).

BẮT BUỘC FORMAT:
- Chỉ xuất 1 JSON object, bắt đầu bằng {{ và kết thúc bằng }}.
- Dùng nháy kép " cho mọi key và string value.
- metaphor_phrases là LIST object đúng dạng:
  {{"phrase":"...","start":0,"end":0}}

Schema mẫu: {json.dumps(schema_hint_annotate, ensure_ascii=False)}
""".strip()

        fewshot_block = ""
        if approach == "few_shot_5" and fewshot_examples:
            ex_parts = []
            for ex in fewshot_examples:
                ex_out = gold_to_annotate_json(ex)
                ex_parts.append("Ví dụ\nCâu: " + ex["sentence"])
                ex_parts.append("JSON:\n" + json.dumps(ex_out, ensure_ascii=False))
            fewshot_block = "\n\n".join(ex_parts)

        user_part = f"Câu: {sentence}\nJSON:"
        if fewshot_block:
            return rules + "\n\n" + fewshot_block + "\n\nBài làm\n" + user_part
        return rules + "\n\n" + user_part

    # ====== JUDGE MODE (Task 4) ======
    gold_interpretation = (gold_interpretation or "").strip()

    rules = f"""
Bạn là người chấm điểm câu diễn giải (paraphrase) tiếng Việt. Trả về DUY NHẤT 1 JSON HỢP LỆ (không markdown, không giải thích, không lặp lại input).

NHIỆM VỤ (TASK 4):
- Chỉ chấm điểm cho GOLD_INTERPRETATION được cung cấp. KHÔNG viết lại, KHÔNG sửa câu.
- Nếu GOLD_INTERPRETATION rỗng => scores=null.

Chấm 8 tiêu chí (0–4):
A) Chất lượng diễn giải: accuracy, clarity, naturalness
B) Tương đồng ngữ nghĩa: meaning, implication, modality, syntax, context

Tính 2 điểm tổng (làm tròn 1 chữ số thập phân):
- overall = (3*accuracy + 2*clarity + 1*naturalness) / 6
- quality = meaning*0.3 + implication*0.25 + modality*0.25 + syntax*0.1 + context*0.1

BẮT BUỘC FORMAT:
- Chỉ xuất 1 JSON object, bắt đầu bằng {{ và kết thúc bằng }}.
- Dùng nháy kép " cho key và string value.
- Output chỉ theo schema: {json.dumps(schema_hint_judge, ensure_ascii=False)}
""".strip()

    fewshot_block = ""
    if approach == "few_shot_5" and fewshot_examples:
        ex_parts = []
        for ex in fewshot_examples:
            ex_out = gold_to_judge_json(ex)
            gi = (ex.get("interpretation") or "").strip() or "<EMPTY>"
            ex_parts.append("Ví dụ\nCâu: " + ex["sentence"])
            ex_parts.append("GOLD_INTERPRETATION: " + gi)
            ex_parts.append("JSON:\n" + json.dumps(ex_out, ensure_ascii=False))
        fewshot_block = "\n\n".join(ex_parts)

    user_part = (
        f"Câu: {sentence}\n"
        f"GOLD_INTERPRETATION: {gold_interpretation if gold_interpretation else '<EMPTY>'}\n"
        f"JSON:"
    )
    if fewshot_block:
        return rules + "\n\n" + fewshot_block + "\n\nBài làm\n" + user_part
    return rules + "\n\n" + user_part

In [9]:
import os
from huggingface_hub import login

login(token="YOUR_HF_TOKEN")

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    trust_remote_code=TRUST_REMOTE_CODE
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=TRUST_REMOTE_CODE,
)
model.eval()
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

print("Loaded:", MODEL_NAME)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded: SeaLLMs/SeaLLMs-v3-7B-Chat


In [11]:
def to_chat_prompt(text_prompt: str) -> str:
    messages = [
        {"role": "system", "content": "Chỉ được xuất ra JSON hợp lệ, không thêm bất kỳ văn bản nào khác."},
        {"role": "user", "content": text_prompt},
    ]
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass

    # fallback Llama-style
    sys = "Chỉ được xuất ra JSON hợp lệ, không thêm bất kỳ văn bản nào khác."
    return f"<s>[INST] <<SYS>>\n{sys}\n<</SYS>>\n\n{text_prompt} [/INST]"

In [12]:
@torch.no_grad()
def generate_batch(prompts: List[str], max_new_tokens: Optional[int] = None) -> List[str]:
    mn = max_new_tokens if max_new_tokens is not None else MAX_NEW_TOKENS

    chat_prompts = [to_chat_prompt(p) for p in prompts]
    inputs = tokenizer(chat_prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    out = model.generate(
        **inputs,
        do_sample=DO_SAMPLE,
        temperature=TEMPERATURE,
        max_new_tokens=mn,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    texts = []
    attn = inputs["attention_mask"]
    for i in range(out.shape[0]):
        prompt_len = int(attn[i].sum().item())
        gen_ids = out[i, prompt_len:]
        texts.append(tokenizer.decode(gen_ids, skip_special_tokens=True))
    return texts

In [13]:
import json, re, ast, os
from typing import Any, Dict, List, Tuple, Optional
from collections import Counter
from tqdm import tqdm

# ====== constants ======
SCORE_KEYS = ["accuracy","clarity","naturalness","meaning","implication","modality","syntax","context","overall","quality"]

# =========================
# JSON extraction utilities
# =========================
def _strip_noise(text: str) -> str:
    t = text.replace("```json", "").replace("```", "")
    t = t.replace("<|im_start|>", "").replace("<|im_end|>", "")

    # ưu tiên bắt JSON output cuối (tránh model echo schema/prompt)
    m = list(re.finditer(r'\{\s*"(have_metaphor|scores)"\s*:', t))
    if m:
        return t[m[-1].start():]

    j = t.rfind("{")
    return t[j:] if j != -1 else t

def _find_top_level_json_objects(s: str) -> List[str]:
    objs, depth, start = [], 0, None
    for i, ch in enumerate(s):
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            if depth > 0:
                depth -= 1
                if depth == 0 and start is not None:
                    objs.append(s[start:i+1])
                    start = None
    return objs

def _repair_span_double_braces(js: str) -> str:
    s = js

    # sửa lỗi }} trước , hoặc ]
    s = re.sub(r"\}\}(?=\s*[,]])", "}", s)

    # sửa lỗi đóng list rồi mới đóng object:  ... ]} , {  ->  ... } , {
    s = re.sub(r"\]\s*\}(?=\s*,\s*\{)", "}", s)

    # (phòng trường hợp ngược lại) ... }], {  ->  ... }, {
    s = re.sub(r"\}\s*\](?=\s*,\s*\{)", "}", s)

    #  bỏ 1 dấu '}' cuối nếu thừa đúng 1 cái
    if s.count("}") == s.count("{") + 1 and s.rstrip().endswith("}"):
        s = s.rstrip()[:-1]

    return s

def _repair_py_literals(js: str) -> str:
    s = re.sub(r"\bNone\b", "null", js)
    s = re.sub(r"\bTrue\b", "true", s)
    s = re.sub(r"\bFalse\b", "false", s)
    s = re.sub(r",(\s*[}\]])", r"\1", s)
    return s

def _parse_one_obj(obj_str: str) -> Optional[Dict[str, Any]]:
    st = _repair_span_double_braces(obj_str.strip())
    st = _repair_unescaped_quotes_in_interpretation(st)
    
    try:
        obj = json.loads(st)
        return obj if isinstance(obj, dict) else None
    except Exception:
        pass

    st2 = _repair_py_literals(st)
    try:
        obj = json.loads(st2)
        return obj if isinstance(obj, dict) else None
    except Exception:
        pass

    try:
        py = st
        py = re.sub(r"\bnull\b", "None", py)
        py = re.sub(r"\btrue\b", "True", py)
        py = re.sub(r"\bfalse\b", "False", py)
        obj = ast.literal_eval(py)
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None
        
def _balance_json_tail(s: str) -> str:
    # thêm ] và } còn thiếu nếu model bị cắt cụt ở cuối
    ob, cb = s.count("{"), s.count("}")
    osq, csq = s.count("["), s.count("]")
    if osq > csq:
        s += "]" * (osq - csq)
    if ob > cb:
        s += "}" * (ob - cb)
    return s

def _repair_unescaped_quotes_in_interpretation(js: str) -> str:
    # chỉ sửa bên trong value của "interpretation": "..."
    m = re.search(r'"interpretation"\s*:\s*"', js)
    if not m:
        return js

    chars = list(js)
    i = m.end()
    esc = False

    while i < len(chars):
        ch = chars[i]
        if esc:
            esc = False
        elif ch == "\\":
            esc = True
        elif ch == '"':
            tail = "".join(chars[i:i+60])
            # gặp quote kết thúc chuỗi interpretation
            if re.match(r'"\s*,\s*"scores"\s*:', tail) or re.match(r'"\s*,\s*}', tail) or re.match(r'"\s*}', tail):
                break
            # quote nằm trong nội dung -> đổi thành nháy đơn
            chars[i] = "'"
        i += 1

    return "".join(chars)
    
def extract_json(text: str) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    t = _strip_noise(text)
    obj_strs = _find_top_level_json_objects(t)
    if not obj_strs:
        i = t.find("{")
        if i != -1:
            cand = _balance_json_tail(t[i:].strip())
            obj = _parse_one_obj(cand)
            if obj is not None:
                return obj, None
        return None, "no_json_object"

    merged, parsed_any = {}, False
    for s in obj_strs:
        obj = _parse_one_obj(s)
        if obj is not None:
            merged.update(obj)
            parsed_any = True

    if not parsed_any:
        return None, "json_load_error: cannot_parse_any_object"

    return merged, None

# =========================
# normalize + deterministic repairs
# =========================
def normalize_annotate(obj: Dict[str, Any]) -> Dict[str, Any]:
    out = dict(obj)

    # map key cũ -> key mới (phòng khi model lỡ trả spans/types)
    if "metaphor_phrases" not in out and "spans" in out:
        mp = []
        for sp in (out.get("spans") or []):
            if isinstance(sp, dict):
                mp.append({
                    "phrase": str(sp.get("phrase","")),
                    "start": sp.get("start", 0),
                    "end": sp.get("end", 0)
                })
        out["metaphor_phrases"] = mp
    if "metaphor_types" not in out and "types" in out:
        out["metaphor_types"] = out.get("types")

    # have_metaphor
    hm = out.get("have_metaphor", 0)
    try:
        hm = int(hm)
    except Exception:
        hm = 0
    hm = 1 if hm == 1 else 0
    out["have_metaphor"] = hm

    # defaults
    if "metaphor_phrases" not in out or not isinstance(out["metaphor_phrases"], list):
        out["metaphor_phrases"] = []
    if "metaphor_types" not in out or not isinstance(out["metaphor_types"], list):
        out["metaphor_types"] = []
    if "interpretation" not in out or not isinstance(out["interpretation"], str):
        out["interpretation"] = ""

    # annotate mode: scores luôn null
    out["scores"] = None

    # have_metaphor=0 => ép rỗng
    if hm == 0:
        out["metaphor_phrases"] = []
        out["metaphor_types"] = []
        out["interpretation"] = ""

    # clean phrases
    clean = []
    for sp in out["metaphor_phrases"]:
        if not isinstance(sp, dict):
            continue
        ph = str(sp.get("phrase", ""))
        if not ph:
            continue
        try:
            st = int(sp.get("start", 0))
            ed = int(sp.get("end", 0))
        except Exception:
            st, ed = 0, 0
        clean.append({"phrase": ph, "start": st, "end": ed})
    out["metaphor_phrases"] = clean

    # clean types theo whitelist
    out["metaphor_types"] = [t for t in out["metaphor_types"] if t in ALLOWED_TYPES]

    return out

def align_phrases_by_phrase(pred: Dict[str, Any], sentence: str) -> Dict[str, Any]:
    """
    Deterministic repair (công bằng):
    - Nếu start/end không khớp substring nhưng phrase xuất hiện trong sentence -> set lại start/end theo find().
    - Nếu phrase không xuất hiện -> drop span đó (không fuzzy để tránh bias).
    """
    if not isinstance(pred, dict):
        return pred
    phs = pred.get("metaphor_phrases", [])
    if not isinstance(phs, list):
        pred["metaphor_phrases"] = []
        return pred

    fixed = []
    for sp in phs:
        if not isinstance(sp, dict):
            continue
        ph = str(sp.get("phrase", ""))
        if not ph:
            continue

        st = sp.get("start", None)
        ed = sp.get("end", None)

        # case 1: đã khớp substring
        if isinstance(st, int) and isinstance(ed, int) and 0 <= st <= ed <= len(sentence) and sentence[st:ed] == ph:
            fixed.append({"phrase": ph, "start": st, "end": ed})
            continue

        # case 2: align bằng find()
        idx = sentence.find(ph)
        if idx != -1:
            fixed.append({"phrase": ph, "start": idx, "end": idx + len(ph)})
            continue

        # case 3: không thấy phrase -> drop
        pass

    pred["metaphor_phrases"] = fixed
    return pred

def normalize_judge(obj: Dict[str, Any]) -> Dict[str, Any]:
    out = dict(obj)

    # wrap scores nếu model trả phẳng
    if "scores" not in out:
        if any(k in out for k in SCORE_KEYS):
            out = {"scores": {k: out.get(k) for k in SCORE_KEYS}}
        else:
            out = {"scores": None}

    sc = out.get("scores", None)
    if sc is None:
        out["scores"] = None
        return out

    if not isinstance(sc, dict):
        out["scores"] = None
        return out

    out["scores"] = {k: sc.get(k) for k in SCORE_KEYS}
    return out

def _round1(x: float) -> float:
    return round(float(x), 1)

def fill_overall_quality_if_missing(j: Dict[str, Any]) -> Dict[str, Any]:
    """
    Nếu model quên overall/quality (null) nhưng có đủ điểm con -> tính lại theo công thức cố định.
    """
    if not isinstance(j, dict) or "scores" not in j:
        return j
    sc = j["scores"]
    if sc is None or not isinstance(sc, dict):
        return j

    # cast string -> float nếu cần
    for k in SCORE_KEYS:
        if k in sc and isinstance(sc[k], str):
            try:
                sc[k] = float(sc[k])
            except Exception:
                pass

    # overall
    if sc.get("overall") is None:
        a, c, n = sc.get("accuracy"), sc.get("clarity"), sc.get("naturalness")
        if isinstance(a,(int,float)) and isinstance(c,(int,float)) and isinstance(n,(int,float)):
            sc["overall"] = _round1((3*a + 2*c + 1*n)/6)

    # quality
    if sc.get("quality") is None:
        m = sc.get("meaning")
        imp = sc.get("implication")
        mod = sc.get("modality")
        syn = sc.get("syntax")
        ctx = sc.get("context")
        if all(isinstance(x,(int,float)) for x in [m, imp, mod, syn, ctx]):
            sc["quality"] = _round1(m*0.3 + imp*0.25 + mod*0.25 + syn*0.1 + ctx*0.1)

    j["scores"] = sc
    return j

# =========================
# validate
# =========================
def validate_annotate(obj: Any, sentence: str):
    need = ["have_metaphor","metaphor_phrases","metaphor_types","interpretation","scores"]
    if not isinstance(obj, dict): return False, "not_dict"
    for k in need:
        if k not in obj: return False, f"missing_{k}"
    if obj["have_metaphor"] not in [0,1]: return False, "bad_have_metaphor"
    if not isinstance(obj["metaphor_phrases"], list): return False, "phrases_not_list"
    if not isinstance(obj["metaphor_types"], list): return False, "types_not_list"
    if not isinstance(obj["interpretation"], str): return False, "interp_not_str"
    if obj["scores"] is not None: return False, "scores_must_be_null_in_annotate"

    for t in obj["metaphor_types"]:
        if t not in ALLOWED_TYPES:
            return False, f"bad_type_{t}"

    # check span substring (sau align thì thường pass)
    for sp in obj["metaphor_phrases"]:
        if not isinstance(sp, dict): return False, "phrase_item_not_dict"
        if not {"phrase","start","end"} <= set(sp.keys()): return False, "phrase_item_missing_keys"
        st, ed, ph = sp["start"], sp["end"], sp["phrase"]
        if not isinstance(st, int) or not isinstance(ed, int): return False, "start_end_not_int"
        if st < 0 or ed < 0 or st > ed or ed > len(sentence): return False, "start_end_out_of_range"
        if sentence[st:ed] != ph: return False, "phrase_not_match_substring"

    return True, None

def validate_judge(obj: Any):
    if not isinstance(obj, dict): return False, "not_dict"
    if "scores" not in obj: return False, "missing_scores"
    sc = obj["scores"]
    if sc is None:
        return True, None
    if not isinstance(sc, dict): return False, "scores_not_dict"

    for k in SCORE_KEYS:
        if k not in sc: return False, f"missing_score_{k}"

    def _is_num(x): return isinstance(x, (int, float)) and (not isinstance(x, bool))

    for k in SCORE_KEYS:
        v = sc[k]
        if v is None: return False, f"score_{k}_is_null"
        if not _is_num(v): return False, f"score_{k}_not_number"
        if v < 0 or v > 4: return False, f"score_{k}_out_of_range"

    return True, None

In [14]:
def run_and_save(dataset, split_name, approach, out_dir, fewshot_examples=None, limit=None):
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{split_name}__{approach}.jsonl")
    data = dataset[:limit] if limit else dataset

    with open(out_path, "w", encoding="utf-8") as f:
        for i in tqdm(range(0, len(data), BATCH_SIZE), desc=f"{split_name}-{approach}"):
            batch = data[i:i+BATCH_SIZE]

            # ===== PASS 1: annotate =====
            prompts_a = [build_prompt(r["sentence"], approach, fewshot_examples, mode="annotate") for r in batch]
            raws_a = generate_batch(prompts_a)

            preds_a, metas_a = [], []
            for r, prompt, raw in zip(batch, prompts_a, raws_a):
                obj, perr = extract_json(raw)
                retried = False
                if obj is None and perr in ["no_json_object", "json_load_error: cannot_parse_any_object"]:
                    raw2 = generate_batch([prompt], max_new_tokens=RETRY_MAX_NEW_TOKENS)[0]
                    obj2, perr2 = extract_json(raw2)
                    if obj2 is not None:
                        obj, perr, raw = obj2, perr2, raw2
                        retried = True

                if obj is None:
                    preds_a.append(None)
                    metas_a.append((False, perr, None, retried, raw))
                else:
                    obj = normalize_annotate(obj)
                    obj = align_phrases_by_phrase(obj, r["sentence"])   # ✅ FIX span indices
                    ok, verr = validate_annotate(obj, r["sentence"])
                    preds_a.append(obj)
                    metas_a.append((ok, perr, verr, retried, raw))

            # ===== PASS 2: judge (chấm GOLD interpretation) =====
            prompts_j, golds = [], []
            for r in batch:
                gi = (r.get("interpretation") or "").strip()
                golds.append(gi)
                if not gi:
                    prompts_j.append(None)
                else:
                    prompts_j.append(build_prompt(r["sentence"], "zero_shot", None, mode="judge", gold_interpretation=gi))

            idx_map = [k for k, p in enumerate(prompts_j) if p is not None]
            raws_j_full = [None] * len(batch)
            if idx_map:
                to_gen = [prompts_j[k] for k in idx_map]
                gen_out = generate_batch(to_gen)
                for k, raw in zip(idx_map, gen_out):
                    raws_j_full[k] = raw

            preds_j, metas_j = [], []
            for r, prompt, raw, gi in zip(batch, prompts_j, raws_j_full, golds):
                if not gi:
                    preds_j.append({"scores": None})
                    metas_j.append((True, None, None, False, None))  # skipped judge
                    continue

                obj, perr = extract_json(raw)
                retried = False
                if obj is None and perr in ["no_json_object", "json_load_error: cannot_parse_any_object"]:
                    raw2 = generate_batch([prompt], max_new_tokens=RETRY_MAX_NEW_TOKENS)[0]
                    obj2, perr2 = extract_json(raw2)
                    if obj2 is not None:
                        obj, perr, raw = obj2, perr2, raw2
                        retried = True

                if obj is None:
                    preds_j.append(None)
                    metas_j.append((False, perr, None, retried, raw))
                else:
                    obj = normalize_judge(obj)
                    obj = fill_overall_quality_if_missing(obj)         # ✅ FIX overall/quality
                    ok, verr = validate_judge(obj)
                    preds_j.append(obj)
                    metas_j.append((ok, perr, verr, retried, raw))

            # ===== save =====
            for r, pa, ma, pj, mj in zip(batch, preds_a, metas_a, preds_j, metas_j):
                ok_a, perr_a, verr_a, retried_a, raw_a = ma
                ok_j, perr_j, verr_j, retried_j, raw_j = mj

                record = {
                    "id": r["id"],
                    "sentence": r["sentence"],
                    "gold": {
                        "have_metaphor": r.get("have_metaphor"),
                        "metaphor_phrases": r.get("metaphor_phrases"),
                        "metaphor_types": r.get("metaphor_types"),
                        "interpretation": r.get("interpretation"),
                        "scores": r.get("scores"),
                    },
                    "pred": pa,
                    "judge": pj,
                    "raw_pred": raw_a,
                    "raw_judge": raw_j,
                    "meta": {
                        "model": MODEL_NAME,
                        "approach": approach,
                        "split": split_name,
                        "temperature": TEMPERATURE,
                        "do_sample": DO_SAMPLE,
                        "max_new_tokens": MAX_NEW_TOKENS,
                        "fewshot_ids": FEWSHOT_IDS if approach == "few_shot_5" else None,

                        "pred_json_valid": ok_a,
                        "pred_parse_error": perr_a,
                        "pred_validate_error": verr_a,
                        "pred_retried": retried_a,

                        "judge_json_valid": ok_j,
                        "judge_parse_error": perr_j,
                        "judge_validate_error": verr_j,
                        "judge_retried": retried_j,
                    }
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print("✅ Saved:", out_path)
    return out_path

def quick_valid_rate(path):
    total, ok_pred, ok_judge = 0, 0, 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            total += 1
            r = json.loads(line)
            if r["meta"].get("pred_json_valid"):
                ok_pred += 1
            if r["meta"].get("judge_json_valid"):
                ok_judge += 1
    return {
        "total": total,
        "pred_valid": (ok_pred, total, ok_pred/total if total else 0.0),
        "judge_valid": (ok_judge, total, ok_judge/total if total else 0.0),
    }

In [15]:
def summarize_invalid(jsonl_path, which="pred", max_show=5):
    """
    which: "pred" hoặc "judge"
    """
    bad = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)
            if which == "pred":
                if not r["meta"].get("pred_json_valid", False):
                    bad.append(r)
            else:
                if not r["meta"].get("judge_json_valid", False):
                    bad.append(r)

    print(f"Invalid({which}):", len(bad))
    if which == "pred":
        print("Top parse_error:", Counter([b["meta"]["pred_parse_error"] for b in bad]).most_common(5))
        print("Top validate_error:", Counter([b["meta"]["pred_validate_error"] for b in bad]).most_common(5))
    else:
        print("Top parse_error:", Counter([b["meta"]["judge_parse_error"] for b in bad]).most_common(5))
        print("Top validate_error:", Counter([b["meta"]["judge_validate_error"] for b in bad]).most_common(5))

    for r in bad[:max_show]:
        print("\n====", r["id"], "====")
        if which == "pred":
            print("parse_error:", r["meta"]["pred_parse_error"])
            print("validate_error:", r["meta"]["pred_validate_error"])
            print("raw_pred:\n", (r.get("raw_pred") or "")[:800])
        else:
            print("parse_error:", r["meta"]["judge_parse_error"])
            print("validate_error:", r["meta"]["judge_validate_error"])
            print("raw_judge:\n", (r.get("raw_judge") or "")[:800])

In [15]:
dev_dir = os.path.join(RESULT_ROOT, "dev")

dev_zero = run_and_save(dev, "dev", "zero_shot", dev_dir, fewshot_examples=None, limit=DEV_LIMIT)
dev_few  = run_and_save(dev, "dev", "few_shot_5", dev_dir, fewshot_examples=fewshot_5, limit=DEV_LIMIT)

print("Dev zero valid:", quick_valid_rate(dev_zero))
print("Dev few  valid:", quick_valid_rate(dev_few))

dev-zero_shot: 100%|██████████| 7/7 [00:38<00:00,  5.44s/it]


✅ Saved: Result/SeaLLMs-v3-7B/dev/dev__zero_shot.jsonl


dev-few_shot_5: 100%|██████████| 7/7 [01:01<00:00,  8.72s/it]

✅ Saved: Result/SeaLLMs-v3-7B/dev/dev__few_shot_5.jsonl
Dev zero valid: {'total': 200, 'pred_valid': (200, 200, 1.0), 'judge_valid': (200, 200, 1.0)}
Dev few  valid: {'total': 200, 'pred_valid': (200, 200, 1.0), 'judge_valid': (200, 200, 1.0)}


In [16]:
summarize_invalid(dev_zero, max_show=5)
summarize_invalid(dev_few,  max_show=5)

Invalid(pred): 0
Top parse_error: []
Top validate_error: []
Invalid(pred): 0
Top parse_error: []
Top validate_error: []


In [17]:
test_dir = os.path.join(RESULT_ROOT, "test")

test_zero = run_and_save(test, "test", "zero_shot", test_dir, fewshot_examples=None, limit=TEST_LIMIT)
test_few  = run_and_save(test, "test", "few_shot_5", test_dir, fewshot_examples=fewshot_5, limit=TEST_LIMIT)

print("Test zero:", test_zero)
print("Test few :", test_few)
print("Test zero valid:", quick_valid_rate(test_zero))
print("Test few  valid:", quick_valid_rate(test_few))

test-zero_shot: 100%|██████████| 54/54 [12:50<00:00, 14.26s/it]


✅ Saved: Result/SeaLLMs-v3-7B/test/test__zero_shot.jsonl


test-few_shot_5: 100%|██████████| 54/54 [21:47<00:00, 24.21s/it]

✅ Saved: Result/SeaLLMs-v3-7B/test/test__few_shot_5.jsonl
Test zero: Result/SeaLLMs-v3-7B/test/test__zero_shot.jsonl
Test few : Result/SeaLLMs-v3-7B/test/test__few_shot_5.jsonl
Test zero valid: {'total': 1701, 'pred_valid': (1700, 1701, 0.9994121105232217), 'judge_valid': (1701, 1701, 1.0)}
Test few  valid: {'total': 1701, 'pred_valid': (1699, 1701, 0.9988242210464433), 'judge_valid': (1701, 1701, 1.0)}


# Fine-Tuning

In [1]:
!pip -q install -U datasets peft trl

In [16]:
import os, json, random
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig

os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
torch.backends.cuda.enable_cudnn_sdp(True)

MODEL_NAME = "SeaLLMs/SeaLLMs-v3-7B-Chat"
DTYPE = torch.bfloat16

RESULT_ROOT = "Result/seallms_v3_7b"
FT_OUT_DIR  = os.path.join(RESULT_ROOT, "sft_multitask")
os.makedirs(FT_OUT_DIR, exist_ok=True)

SYSTEM_JSON_ONLY = "Chỉ được xuất ra JSON hợp lệ, không thêm bất kỳ văn bản nào khác."

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=DTYPE,
)

tokenizer_train = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer_train.pad_token is None:
    tokenizer_train.pad_token = tokenizer_train.eos_token
tokenizer_train.padding_side = "right"

model_train = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_cfg,
    torch_dtype=DTYPE,
    attn_implementation="sdpa",
)
model_train.config.use_cache = False

peft_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

print("Loaded SeaLLMs train model + tokenizer.")
print("EOS token:", repr(tokenizer_train.eos_token), " | eos_token_id:", tokenizer_train.eos_token_id)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded SeaLLMs train model + tokenizer.
EOS token: '<|im_end|>'  | eos_token_id: 151645


In [17]:
def make_prompt_only(tokenizer, text_prompt: str) -> str:
    msgs = [
        {"role": "system", "content": SYSTEM_JSON_ONLY},
        {"role": "user", "content": text_prompt},
    ]
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    # fallback (rare)
    sys = SYSTEM_JSON_ONLY
    return f"<s>[INST] <<SYS>>\n{sys}\n<</SYS>>\n\n{text_prompt} [/INST]"

def make_dataset_prompt_completion(rows, tokenizer, p_judge=1.0, seed=42):
    rnd = random.Random(seed)
    data = []
    EOS = tokenizer.eos_token or ""  # <-- SeaLLMs EOS

    for r in rows:
        sent = r["sentence"]

        # 1) ANNOTATE
        prompt_a = build_prompt(sent, "zero_shot", fewshot_examples=None, mode="annotate")
        prompt_a = make_prompt_only(tokenizer, prompt_a)
        completion_a = json.dumps(gold_to_annotate_json(r), ensure_ascii=False) + EOS
        data.append({"prompt": prompt_a, "completion": completion_a, "task": "annotate"})

        # 2) JUDGE (chỉ khi có interpretation + scores thật)
        if rnd.random() <= p_judge:
            gi = (r.get("interpretation") or "").strip()
            has_scores = bool(r.get("has_scores", False)) and (r.get("scores") is not None)
            if gi and has_scores:
                prompt_j = build_prompt(sent, "zero_shot", fewshot_examples=None, mode="judge", gold_interpretation=gi)
                prompt_j = make_prompt_only(tokenizer, prompt_j)
                completion_j = json.dumps(gold_to_judge_json(r), ensure_ascii=False) + EOS
                data.append({"prompt": prompt_j, "completion": completion_j, "task": "judge"})

    rnd.shuffle(data)
    return Dataset.from_list(data)

# NOTE: dùng tokenizer_train (SeaLLMs) để build
train_sft = make_dataset_prompt_completion(train, tokenizer_train, p_judge=1.0, seed=42)
dev_sft   = make_dataset_prompt_completion(dev,   tokenizer_train, p_judge=1.0, seed=43)

print(train_sft)
print("Train task counts:", {t: train_sft["task"].count(t) for t in set(train_sft["task"])})
print("Dev task counts  :", {t: dev_sft["task"].count(t) for t in set(dev_sft["task"])})
print("\nSample prompt head:\n", train_sft[0]["prompt"][:300])
print("\nSample completion:\n", train_sft[0]["completion"][:200])

Dataset({
    features: ['prompt', 'completion', 'task'],
    num_rows: 8156
})
Train task counts: {'judge': 2206, 'annotate': 5950}
Dev task counts  : {'judge': 317, 'annotate': 850}

Sample prompt head:
 <|im_start|>system
Chỉ được xuất ra JSON hợp lệ, không thêm bất kỳ văn bản nào khác.<|im_end|>
<|im_start|>user
Bạn là người gán nhãn ẩn dụ tiếng Việt. Trả về DUY NHẤT 1 JSON HỢP LỆ (không markdown, không giải thích, không lặp lại input).

ĐỊNH NGHĨA NGẮN:
Ẩn dụ là cách diễn đạt dựa trên ánh xạ khái

Sample completion:
 {"have_metaphor": 1, "metaphor_phrases": [{"phrase": "Tóc tang phủ ngập làng quê", "start": 0, "end": 26}, {"phrase": "Trăng Thu đã chênh chếch núi", "start": 27, "end": 55}], "metaphor_types": ["emot


In [18]:
import trl
from trl import SFTTrainer, SFTConfig

print("trl version:", trl.__version__)

FIELDS = set(SFTConfig.__dataclass_fields__.keys())
def mk_sft_config(**kw):
    kw = {k: v for k, v in kw.items() if k in FIELDS}
    return SFTConfig(**kw)

max_kw = {}
if "max_length" in FIELDS:
    max_kw["max_length"] = 2048
elif "max_seq_length" in FIELDS:
    max_kw["max_seq_length"] = 2048

maybe_eos = {}
if "eos_token" in FIELDS:
    maybe_eos["eos_token"] = tokenizer_train.eos_token

sft_args = mk_sft_config(
    output_dir=FT_OUT_DIR,
    learning_rate=1e-4,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_ratio=0.03,
    logging_steps=10,
    save_steps=200,
    eval_steps=200,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    evaluation_strategy="steps", 
    eval_strategy="steps", 
    **max_kw,
)


trainer = SFTTrainer(
    model=model_train,
    args=sft_args,
    train_dataset=train_sft,
    eval_dataset=dev_sft,
    peft_config=peft_cfg,
    processing_class=tokenizer_train,
)

trainer.train()
trainer.save_model(FT_OUT_DIR)
tokenizer_train.save_pretrained(FT_OUT_DIR)
print("✅ Saved FT adapter to:", FT_OUT_DIR)

trl version: 0.26.2


Adding EOS to train dataset:   0%|          | 0/8156 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8156 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8156 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1167 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1167 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1167 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,0.212600,0.216630,1.801447,2306416.000000,0.938812
400,0.207300,0.203431,1.814341,4609104.000000,0.943160
600,0.152400,0.205830,1.680824,6899748.000000,0.942634
800,0.149600,0.201399,1.632511,9202394.000000,0.943959
1000,0.134600,0.199807,1.612284,11503686.000000,0.944871


✅ Saved FT adapter to: Result/seallms_v3_7b/sft_multitask


In [19]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

model = AutoPeftModelForCausalLM.from_pretrained(
    FT_OUT_DIR,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(FT_OUT_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Loaded FT model for inference.")

# ===== DEV FT =====
dev_dir_ft = os.path.join(RESULT_ROOT, "dev_ft")
dev_ft_zero = run_and_save(dev, "dev", "ft_zero_shot", dev_dir_ft, fewshot_examples=None, limit=DEV_LIMIT)
dev_ft_few  = run_and_save(dev, "dev", "ft_few_shot_5", dev_dir_ft, fewshot_examples=fewshot_5, limit=DEV_LIMIT)

print("Dev FT zero valid:", quick_valid_rate(dev_ft_zero))
print("Dev FT few  valid:", quick_valid_rate(dev_ft_few))

# ===== TEST FT =====
test_dir_ft = os.path.join(RESULT_ROOT, "test_ft")
test_ft_zero = run_and_save(test, "test", "ft_zero_shot", test_dir_ft, fewshot_examples=None, limit=TEST_LIMIT)
test_ft_few  = run_and_save(test, "test", "ft_few_shot_5", test_dir_ft, fewshot_examples=fewshot_5, limit=TEST_LIMIT)

print("Test FT zero valid:", quick_valid_rate(test_ft_zero))
print("Test FT few  valid:", quick_valid_rate(test_ft_few))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded FT model for inference.


dev-ft_zero_shot: 100%|██████████| 7/7 [01:47<00:00, 15.42s/it]


✅ Saved: Result/seallms_v3_7b/dev_ft/dev__ft_zero_shot.jsonl


dev-ft_few_shot_5: 100%|██████████| 7/7 [01:48<00:00, 15.43s/it]


✅ Saved: Result/seallms_v3_7b/dev_ft/dev__ft_few_shot_5.jsonl
Dev FT zero valid: {'total': 200, 'pred_valid': (200, 200, 1.0), 'judge_valid': (200, 200, 1.0)}
Dev FT few  valid: {'total': 200, 'pred_valid': (200, 200, 1.0), 'judge_valid': (200, 200, 1.0)}


test-ft_zero_shot: 100%|██████████| 54/54 [17:58<00:00, 19.98s/it]


✅ Saved: Result/seallms_v3_7b/test_ft/test__ft_zero_shot.jsonl


test-ft_few_shot_5: 100%|██████████| 54/54 [18:07<00:00, 20.13s/it]

✅ Saved: Result/seallms_v3_7b/test_ft/test__ft_few_shot_5.jsonl
Test FT zero valid: {'total': 1701, 'pred_valid': (1701, 1701, 1.0), 'judge_valid': (1701, 1701, 1.0)}
Test FT few  valid: {'total': 1701, 'pred_valid': (1701, 1701, 1.0), 'judge_valid': (1701, 1701, 1.0)}
